# AEGIS — entraînement NER PII sur **Google Colab**

Pipeline aligné sur [`training/README.md`](https://github.com/zokastech/aegis/blob/main/training/README.md) : données synthétiques EU (IOB2), fine-tuning **XLM-RoBERTa-base**, export **ONNX** pour `aegis-ner`.

**Avant de lancer** : dans Colab, menu *Exécution* → *Modifier le type d'exécution* → **T4 GPU** (ou mieux) pour un entraînement raisonnable.

**Sécurité** : n’utilisez que des données synthétiques ou anonymisées ; ne collez pas de vraies PII.

Licence : Apache-2.0 / MIT — [zokastech.fr](https://zokastech.fr).

## 1. GPU et clone du dépôt

Par défaut le notebook clone `zokastech/aegis` dans `/content/aegis`. Pour une autre branche : définissez `BRANCH` ci-dessous.

In [ ]:
# @title Vérifier le GPU
!nvidia-smi 2>/dev/null || echo "Pas de GPU NVIDIA — passez en runtime GPU ou l'entraînement sera très lent sur CPU."

In [ ]:
# @title Cloner AEGIS (branche main)
import os
from pathlib import Path

BRANCH = "main"  # @param {type:"string"}
REPO_URL = "https://github.com/zokastech/aegis.git"
ROOT = Path("/content")
REPO_ROOT = ROOT / "aegis"

if REPO_ROOT.is_dir():
    !cd {REPO_ROOT} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_ROOT}

%cd {REPO_ROOT}
assert (REPO_ROOT / "training" / "dataset_builder.py").is_file(), "Clone incomplet : training/dataset_builder.py manquant"
print("REPO_ROOT =", REPO_ROOT.resolve())

In [ ]:
# @title Dépendances Python (training)
import subprocess
import sys

req = REPO_ROOT / "training" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
print("OK —", sys.executable)

In [ ]:
# @title Chemins et PYTHONPATH
import os
import sys
from pathlib import Path

TRAINING_DIR = REPO_ROOT / "training"
DATA_DIR = TRAINING_DIR / "data"
OUTPUT_DIR = TRAINING_DIR / "outputs"
EXPORT_DIR = TRAINING_DIR / "exports" / "onnx_colab"

os.chdir(REPO_ROOT)
sys.path.insert(0, str(TRAINING_DIR))

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TRAINING_DIR =", TRAINING_DIR)

## 2. Jeu synthétique (IOB2, multilingue UE)

`NUM_SYNTHETIC` petit = démo rapide. Pour viser une meilleure qualité, montez à **50_000** (temps + mémoire plus élevés).

In [ ]:
from dataset_builder import build_dataset

NUM_SYNTHETIC = 5000  # @param {type:"integer"}
SEED = 42
SYNTH_PATH = DATA_DIR / "colab_eu_pii_synthetic"

SYNTH_PATH.parent.mkdir(parents=True, exist_ok=True)
ds_synth = build_dataset(NUM_SYNTHETIC, seed=SEED)
ds_synth.save_to_disk(str(SYNTH_PATH))
print(ds_synth)
print("Sauvegardé:", SYNTH_PATH)

## 3. Fusion avec des JSONL du dépôt (optionnel)

Si les fichiers d’exemple existent dans le clone, on fusionne avec le synthétique. Sinon on entraîne **uniquement** sur le jeu synthétique.

In [ ]:
import subprocess
import sys

CUSTOM_JSONL = [
    REPO_ROOT / "datasets" / "training" / "ner_custom" / "samples.jsonl",
    REPO_ROOT / "datasets" / "training" / "ner_financial_seed" / "samples.jsonl",
]
existing = [p for p in CUSTOM_JSONL if p.is_file()]
CUSTOM_HF_PATH = DATA_DIR / "colab_from_jsonl"
MERGED_PATH = DATA_DIR / "colab_merged_synth_custom"
DATASET_FOR_TRAIN = SYNTH_PATH

if existing:
    subprocess.check_call(
        [
            sys.executable,
            str(TRAINING_DIR / "jsonl_to_hf_dataset.py"),
            *[str(p) for p in existing],
            "--output",
            str(CUSTOM_HF_PATH),
            "--val_ratio",
            "0.2",
            "--seed",
            str(SEED),
        ],
        cwd=str(TRAINING_DIR),
    )
    subprocess.check_call(
        [
            sys.executable,
            str(TRAINING_DIR / "merge_hf_datasets.py"),
            str(SYNTH_PATH),
            str(CUSTOM_HF_PATH),
            "--output",
            str(MERGED_PATH),
        ],
        cwd=str(TRAINING_DIR),
    )
    DATASET_FOR_TRAIN = MERGED_PATH
    print("Jeu fusionné:", MERGED_PATH)
else:
    print("Aucun JSONL d’exemple — entraînement sur synthétique seul:", SYNTH_PATH)

## 4. Fine-tuning (`train_ner.py`)

Sur **CUDA** (Colab GPU) : `--fp16` est activé automatiquement. En **CPU** : batch réduit et pas de fp16.

Si **CUDA out of memory** : baissez `BATCH` (ex. 4) ou augmentez `GRAD_ACCUM`, ou `MAX_SEQ` (ex. 256).

In [ ]:
import importlib.util
import os
import subprocess
import sys

for _pkg in ("seqeval", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

import torch

MODEL_OUT = OUTPUT_DIR / "ner-colab-xlmr"
MODEL_OUT.mkdir(parents=True, exist_ok=True)

use_cuda = torch.cuda.is_available()
use_cpu_flag = os.environ.get("AEGIS_TRAIN_CPU", "").lower() in ("1", "true", "yes") or not use_cuda

if use_cuda:
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", "8"))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", "1"))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", "512"))
    FP16 = True
else:
    BATCH = int(os.environ.get("AEGIS_TRAIN_BATCH", "4"))
    GRAD_ACCUM = int(os.environ.get("AEGIS_TRAIN_GRAD_ACCUM", "2"))
    MAX_SEQ = int(os.environ.get("AEGIS_MAX_SEQ_LENGTH", "256"))
    FP16 = False

NUM_EPOCHS = 2  # @param {type:"integer"}

train_cmd = [
    sys.executable,
    str(TRAINING_DIR / "train_ner.py"),
    "--dataset",
    str(DATASET_FOR_TRAIN),
    "--output_dir",
    str(MODEL_OUT),
    "--num_train_epochs",
    str(NUM_EPOCHS),
    "--per_device_train_batch_size",
    str(BATCH),
    "--per_device_eval_batch_size",
    str(min(BATCH, 8)),
    "--gradient_accumulation_steps",
    str(GRAD_ACCUM),
    "--max_seq_length",
    str(MAX_SEQ),
    "--gradient_checkpointing",
]
if use_cpu_flag:
    train_cmd.append("--cpu")
if FP16 and not use_cpu_flag:
    train_cmd.append("--fp16")

print(
    "Device:",
    "CUDA + fp16" if (use_cuda and FP16) else ("CPU" if use_cpu_flag else "MPS/other"),
    "| batch",
    BATCH,
    "| grad_accum",
    GRAD_ACCUM,
    "| max_seq",
    MAX_SEQ,
)
print(" ".join(train_cmd))
subprocess.check_call(train_cmd, cwd=str(TRAINING_DIR))

## 5. Hugging Face Hub (optionnel)

1. Créez un token sur [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
2. Dans Colab : *Secrets* (icône clé) → `HF_TOKEN`, ou `huggingface-cli login` dans une cellule.
3. Renseignez `HF_REPO_ID` (ex. `votre_org/aegis-ner-eu-pii`).

In [ ]:
import os
import subprocess
import sys

HF_REPO_ID = ""  # @param {type:"string"}
HF_PRIVATE = False  # @param {type:"boolean"}

if not HF_REPO_ID.strip():
    HF_REPO_ID = os.environ.get("AEGIS_HF_REPO_ID", "").strip()

BEST_HF = MODEL_OUT / "best_hf"
if not HF_REPO_ID:
    print("Aucun HF_REPO_ID — étape ignorée.")
elif not BEST_HF.is_dir() or not (BEST_HF / "config.json").is_file():
    print("best_hf introuvable — lancez d’abord l’entraînement (section 4).")
else:
    push_cmd = [
        sys.executable,
        str(TRAINING_DIR / "push_hf_model.py"),
        "--model_dir",
        str(BEST_HF),
        "--repo_id",
        HF_REPO_ID,
    ]
    if HF_PRIVATE:
        push_cmd.append("--private")
    subprocess.check_call(push_cmd, cwd=str(TRAINING_DIR))
    print("Hub:", f"https://huggingface.co/{HF_REPO_ID}")

## 6. Export ONNX (`aegis-ner`)

Sortie dans `training/exports/onnx_colab/` : `model.onnx`, `model_int8.onnx`, `tokenizer_hf/tokenizer.json`.

In [ ]:
import subprocess
import sys

BEST_HF = MODEL_OUT / "best_hf"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if not BEST_HF.is_dir():
    raise FileNotFoundError(f"Checkpoint manquant: {BEST_HF}")

export_cmd = [
    sys.executable,
    str(TRAINING_DIR / "export_onnx.py"),
    "--model_dir",
    str(BEST_HF),
    "--out_dir",
    str(EXPORT_DIR),
]
print(" ".join(export_cmd))
subprocess.check_call(export_cmd, cwd=str(TRAINING_DIR))
print("Export terminé:", EXPORT_DIR)

## 7. Sauvegarder sur Google Drive (optionnel)

Les fichiers sous `/content` sont **perdus** quand la session Colab se termine. Montez Drive et copiez le dossier `best_hf` et/ou l’export ONNX.

In [ ]:
# @title Monter Drive et copier les artefacts
from pathlib import Path

SAVE_TO_DRIVE = False  # @param {type:"boolean"}
DRIVE_SUBFOLDER = "AEGIS_NER_colab"  # @param {type:"string"}

if not SAVE_TO_DRIVE:
    print("SAVE_TO_DRIVE=False — rien à copier.")
else:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    dest_root = Path("/content/drive/MyDrive") / DRIVE_SUBFOLDER
    dest_root.mkdir(parents=True, exist_ok=True)

    if MODEL_OUT.joinpath("best_hf").is_dir():
        shutil.copytree(MODEL_OUT / "best_hf", dest_root / "best_hf", dirs_exist_ok=True)
    if EXPORT_DIR.is_dir():
        shutil.copytree(EXPORT_DIR, dest_root / "onnx_export", dirs_exist_ok=True)
    print("Copié vers:", dest_root)

## Références

- [`training/README.md`](https://github.com/zokastech/aegis/blob/main/training/README.md)
- Notebook local : [`train_ner_pii.ipynb`](train_ner_pii.ipynb)
- Jeux HF publics : [`train_ner_hf_public.ipynb`](train_ner_hf_public.ipynb)
- Rapport d’évaluation : `python training/evaluate.py --dataset … --model_dir …/best_hf --out_report report.html`